**Downloading data set for building dataset for book description**

In [3]:
import requests, time, json

subjects = ["fantasy","science_fiction","mystery","romance","history",
            "biography","psychology","programming","cooking","philosophy"]
books, seen = [], set()
headers = {"User-Agent": "semantic-search-learning (your_email@example.com)"}

for subj in subjects:
    r = requests.get(f"https://openlibrary.org/subjects/{subj}.json",
                     params={"limit": 100}, headers=headers, timeout=30)
    works = r.json().get("works", [])
    print(subj, "works found:", len(works))
    for w in works:
        if w["key"] in seen:
            continue
        try:
            d = requests.get(f"https://openlibrary.org{w['key']}.json",
                             headers=headers, timeout=30).json()
        except Exception as e:
            print("skip", w["key"], e)
            continue
        desc = d.get("description", "")
        if isinstance(desc, dict):
            desc = desc.get("value", "")
        if len(desc) > 150:
            seen.add(w["key"])
            books.append({"id": w["key"], "title": w["title"],
                          "authors": [a["name"] for a in w.get("authors", [])],
                          "category": subj, "description": desc})
        time.sleep(0.2)
    print("total so far:", len(books))

json.dump(books, open("books.json", "w"), indent=2)
print(len(books))

fantasy works found: 100
total so far: 80
science_fiction works found: 100
total so far: 152
mystery works found: 100
total so far: 236
romance works found: 100
total so far: 320
history works found: 100
total so far: 395
biography works found: 100
total so far: 460
psychology works found: 100
total so far: 524
programming works found: 100
total so far: 556
cooking works found: 100
total so far: 589
philosophy works found: 100
total so far: 645
645


In [5]:
with open("books.json", "r") as f:
    books = json.load(f)

print(len(books))


645


**Text precessing and dataset split**

In [10]:
description =[]
for books_detail in books:
  description.append(books_detail["title"] + ". " + books_detail["description"])

print(description)


["Alice's Adventures in Wonderland. Alice's Adventures in Wonderland (commonly Alice in Wonderland) is an 1865 English children's novel by Lewis Carroll. A young girl named Alice falls through a rabbit hole into a fantasy world of anthropomorphic creatures. It is seen as an example of the literary nonsense genre.\r\n\r\nOne of the best-known works of Victorian literature, its narrative, structure, characters and imagery have had huge influence on popular culture and literature, especially in the fantasy genre.", 'The Wonderful Wizard of Oz. Over a century after its initial publication, The Wonderful Wizard of Oz is still captivating the hearts of countless readers. Come adventure with Dorothy and her three friends: the Scarecrow, the Tin Woodman, and the Cowardly Lion, as they follow the Yellow Brick Road to the Emerald City for an audience with the Great Oz, the mightiest Wizard in the land, and the only one that can return Dorothy to her home in Kansas.', 'Treasure Island. Traditiona

In [12]:
import re, html
# cut everything after these markers
CUT = re.compile(
    r"(\n\s*-{3,}.*|\n\s*(Also contained in|Contained in|Contains|See also)\s*:.*)",
    re.I | re.S)

BOILER = [
    r"Please Note:.*",                         # ebook boilerplate
    r"E-book exclusive extras.*?(?=[A-Z][a-z])",
    r"\(\s*source\s*\)|\(\s*\[source\].*?\)",
    r"\(Wikipedia\)|\(Source:.*?\)",
    r"--\s*(back cover|book jacket|container|jacket)\.?",
]

def clean(text):
    t = html.unescape(text)
    t = CUT.sub("", t)                                  # drop link-list sections
    t = re.sub(r"<[^>]+>", " ", t)                      # strip HTML tags
    t = re.sub(r"^\s*\[\d+\]:.*$", "", t, flags=re.M)   # [1]: http://...
    t = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", t)      # [text](url) -> text
    t = re.sub(r"\[([^\]]+)\]\[\d+\]", r"\1", t)        # [text][1] -> text
    t = re.sub(r"https?://\S+", "", t)                  # bare URLs
    t = re.sub(r"[*_#>`~]+", "", t)                     # markdown symbols
    for b in BOILER:
        t = re.sub(b, "", t, flags=re.I | re.S)
    t = re.sub(r"\s+", " ", t).strip()                  # \r\n, tabs, double spaces
    return t

dataset = [clean(x) for x in  description]


In [14]:
print(len(dataset))

645


**genentration embedding from the clean list and test it**

In [21]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(description, convert_to_tensor=True)
print(embeddings.shape)

torch.Size([645, 384])


In [33]:
from sentence_transformers import util
import random

x = random.randint(0, 644)
print(f"Actual string- {description[x]}")


hits = util.semantic_search(embeddings[x], embeddings, top_k=5)[0]
for h in hits:
    print("\nSearch string -"+description[h["corpus_id"]], round(h["score"], 3))

Actual string- Prester John. I mind as if it were yesterday my first sight of the man. Little I knew at the time how big the moment was with destiny, or how often that face seen in the fitful moonlight would haunt my sleep and disturb my waking hours. But I mind yet the cold grue of terror I got from it, a terror which was surely more than the due of a few truant lads breaking the Sabbath with their play.

Search string -Prester John. I mind as if it were yesterday my first sight of the man. Little I knew at the time how big the moment was with destiny, or how often that face seen in the fitful moonlight would haunt my sleep and disturb my waking hours. But I mind yet the cold grue of terror I got from it, a terror which was surely more than the due of a few truant lads breaking the Sabbath with their play. 1.0

Search string -The Man Who Was Thursday. Can you trust yourself when you don't know who you are? In a park in London, secret policeman Gabriel Syme strikes up a conversation wi